# Day 4

## Tokenizing with code

In [ ]:
#Check available encodings

try:
    import tiktoken
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken"])
    import tiktoken

print(sorted(tiktoken.list_encoding_names()))

['cl100k_base', 'gpt2', 'o200k_base', 'o200k_harmony', 'p50k_base', 'p50k_edit', 'r50k_base']


In [ ]:
import tiktoken

#cl100k_base works for GPT-4, GPT-4.1 family and GPT-3.5
encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode("Hi my name is Supriti and I like to travel")

In [9]:
tokens

[13347, 856, 836, 374, 6433, 1018, 72, 323, 358, 1093, 311, 5944]

In [10]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

13347 = Hi
856 =  my
836 =  name
374 =  is
6433 =  Sup
1018 = rit
72 = i
323 =  and
358 =  I
1093 =  like
311 =  to
5944 =  travel


In [12]:
encoding.decode([374])

' is'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [13]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


##

_I'm creating a new instance of the Groq Python Client library, a lightweight wrapper around making HTTP calls to an endpoint_

In [16]:
#Syntax for Groq

from groq import Groq

client = Groq()

### A message to OpenAI is a list of dicts

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Supriti!"}
    ]

In [20]:
response = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
response.choices[0].message.content

"Hello Supriti, it's nice to meet you. Is there something I can help you with or would you like to chat?"

### OK let's now ask a follow-up question

In [21]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [23]:
response = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
response.choices[0].message.content

"I don't know your name. I'm a text-based AI assistant, and we've just started our conversation. I don't have any prior knowledge about you, including your name. If you'd like to share your name, I'd be happy to know it, though."

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [26]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Supriti!"},
    {"role": "assistant", "content": "Hi Supriti! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [27]:
response = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
response.choices[0].message.content

"Your name is Supriti. It's nice to meet you."

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The Groq(or other LLMs) product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!



## Context Window

Max number of tokens that the model can consider when generating the next token. 

It includes the original input prompt, subsequent conversation, the latest input prompt and almost all the output prompt.

It governs how well the model can remember references, content and context.

check out - https://www.vellum.ai/llm-leaderboard